In [75]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

sns.set_style("whitegrid")

import matplotlib

# set font size to 16
matplotlib.rcParams.update({"font.size": 16})

SEED = 4

In [76]:
n_runs = 100
# ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
start_type = "Syn"

#Use the csv files again
#Load accuracy, precision, recall and f1 score
def load_dict_from_csv(filename):
    df = pd.read_csv(filename)
    nested_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        nested_dict.setdefault(run, {})[method] = group['Value'].tolist()
    return nested_dict

#Load time measurements
def load_time_dict_from_csv(filename):
    df = pd.read_csv(filename)
    time_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        pairs = group[['Selection_Time', 'Train_Time']].values.tolist()
        time_dict.setdefault(run, {})[method] = pairs
    return time_dict

#Load type distributions
def load_distributions_dict_from_csv(filename, all_labels):
    df = pd.read_csv(filename)
    dist_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        arrs = group[all_labels].values.tolist()
        dist_dict.setdefault(run, {})[method] = arrs
    return dist_dict

#Load benchmark values
def load_benchmark_dict_from_csv(filename):
    df = pd.read_csv(filename)
    return {col: df[col].dropna().tolist() for col in df.columns}

def get_summary(loaded_dict):
    methods = list(loaded_dict[0].keys())
    summary_dict = {}
    for method in methods:
        all_runs = np.array([loaded_dict[run][method] for run in range(n_runs)]) # shape: (100, array_length)
        # Compute statistics
        mean_values = np.mean(all_runs, axis=0)
        plow_values = np.percentile(all_runs, 10, axis=0)
        phigh_values = np.percentile(all_runs, 90, axis=0)
        # Store in nested structure for easy plotting
        summary_dict[method] = {
            "mean": mean_values.tolist(),
            "plow": plow_values.tolist(),
            "phigh": phigh_values.tolist()
        }
    return summary_dict

benchmark_dict_loaded = load_benchmark_dict_from_csv(f"./results_dict_new/benchmark_dict_{start_type}_{n_runs}.csv")
accuracy_dict_loaded = load_dict_from_csv( f"./results_dict_new/accuracy_dict_{start_type}_{n_runs}.csv")
precision_dict_loaded = load_dict_from_csv( f"./results_dict_new/precision_dict_{start_type}_{n_runs}.csv")
recall_dict_loaded = load_dict_from_csv( f"./results_dict_new/recall_dict_{start_type}_{n_runs}.csv")
f1score_dict_loaded = load_dict_from_csv( f"./results_dict_new/f1score_dict_{start_type}_{n_runs}.csv")
time_dict_loaded = load_time_dict_from_csv(f"./results_dict_new/time_dict_{start_type}_{n_runs}.csv")
all_labels = ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
distributions_dict_loaded = load_distributions_dict_from_csv(f"./results_dict_new/distributions_dict_{start_type}_{n_runs}.csv", all_labels)

# Mean Benchmark Values
benchmark_means_dict = {k: np.mean(v) for k, v in benchmark_dict_loaded.items()}
#Mean Accuracy Values
accuracy_summary = get_summary(accuracy_dict_loaded)
# Mean Precision Values
precision_summary = get_summary(precision_dict_loaded)
# Mean Recall Values
recall_summary =  get_summary(recall_dict_loaded)
# Mean F1 Score Values
f1score_summary = get_summary(f1score_dict_loaded)



In [77]:
#Calculate distributions at specific f1 score thresholds
f1score_dict_loaded = load_dict_from_csv( f"./results_dict/f1score_dict_{start_type}_{n_runs}.csv")
results = {}
for run in range(len(f1score_dict_loaded)):
    for method in f1score_dict_loaded[run].keys():
        if len(f1score_dict_loaded[run][method]) == 351:
            f1score_dict_loaded[run][method] = np.array(f1score_dict_loaded[run][method])[::10]
            f1score_dict_loaded[run][method] = np.array(f1score_dict_loaded[run][method])[1:]
        else:
            f1score_dict_loaded[run][method] = np.array(f1score_dict_loaded[run][method])[1:]

for run in range(n_runs):
    run_data = f1score_dict_loaded[run]
    methods = list(run_data.keys())

    # Flatten per method into (method, index, value)
    f1_events = []
    for method in methods:
        values = np.array(run_data[method])
        for i, v in enumerate(values):
            f1_events.append((method, i, v))
    
    # Sort events by iteration (index)
    f1_events.sort(key=lambda x: x[1])
    
    # Track first >=0.95 and first/second >=0.99
    first_95 = None
    first_99 = None
    second_99 = None
    third_99 = None
    seen_99_methods = set()
    
    for method, idx, val in f1_events:
        if first_95 is None and val >= 0.95:
            first_95 = (method, idx)
        if val >= 0.99 and method not in seen_99_methods:
            seen_99_methods.add(method)
            if first_99 is None:
                first_99 = (method, idx)
            elif second_99 is None:
                second_99 = (method, idx)
            elif third_99 is None:
                third_99 = (method, idx)
        if first_95 and first_99 and second_99 and third_99:
            break

    results[run] = {
        "first_95_method": first_95[0] if first_95 else None,
        "first_95_index": first_95[1] if first_95 else None,
        "first_99_method": first_99[0] if first_99 else None,
        "first_99_index": first_99[1] if first_99 else None,
        "second_99_method": second_99[0] if second_99 else None,
        "second_99_index": second_99[1] if second_99 else None,
        "third_99_index": third_99[1] if third_99 else None,
        "third_99_method": third_99[0] if third_99 else None
    }


# Ensure consistent method list
methods = list(distributions_dict_loaded[0].keys())
# Containers for distributions at each threshold
all_distr_95 = {method: [] for method in methods}
all_distr_99_1 = {method: [] for method in methods}
all_distr_99_2 = {method: [] for method in methods}
all_distr_99_3 = {method: [] for method in methods}

for run in range(n_runs):
    idx_1 = results[run]["first_95_index"]
    idx_2 = results[run]["first_99_index"]
    idx_3 = results[run]["second_99_index"]
    idx_4 = results[run]["third_99_index"]

    for method in methods:
        dist_arr = np.array(distributions_dict_loaded[run][method])
        n_points = dist_arr.shape[0]

        # Determine whether method is batch or single-step
        if n_points == 350:
            i_1 = ((idx_1 + 1) * 10) - 1
            i_2 = ((idx_2 + 1) * 10) - 1
            i_3 = ((idx_3 + 1) * 10) - 1 
            i_4 = ((idx_4 + 1) * 10) - 1
        else:
            i_1 = idx_1
            i_2 = idx_2
            i_3 = idx_3
            i_4 = idx_4

        all_distr_95[method].append(dist_arr[i_1])
        all_distr_99_1[method].append(dist_arr[i_2])
        all_distr_99_2[method].append(dist_arr[i_3])
        all_distr_99_3[method].append(dist_arr[i_4])

# Mean Distributions
mean_distr_95 = {m: np.mean(all_distr_95[m], axis=0) for m in methods}
mean_distr_99_1 = {m: np.mean(all_distr_99_1[m], axis=0) for m in methods}
mean_distr_99_2 = {m: np.mean(all_distr_99_2[m], axis=0) for m in methods}
mean_distr_99_3 = {m: np.mean(all_distr_99_3[m], axis=0) for m in methods}

In [78]:
#Get the mean sorting
results_summary = {}
f1score_summary_mean = defaultdict(lambda: None)
for method in f1score_summary.keys():
    if len(f1score_summary[method]["mean"]) == 351:
        f1score_summary_mean[method]= np.array(f1score_summary[method]["mean"])[::10]
        f1score_summary_mean[method] = np.array(f1score_summary_mean[method])[1:]
    else:
        f1score_summary_mean[method] = np.array(f1score_summary[method]["mean"])[1:]



methods = list(f1score_summary_mean.keys())

# Flatten per method into (method, index, value)
f1_events = []
for method in methods:
    values = np.array(f1score_summary_mean[method])
    for i, v in enumerate(values):
        f1_events.append((method, i, v))
    
# Sort events by iteration (index)
f1_events.sort(key=lambda x: x[1])
    
# Track first >=0.95 and first/second >=0.99
first_95 = None
first_99 = None
second_99 = None
third_99 = None
seen_99_methods = set()
    
for method, idx, val in f1_events:
    if first_95 is None and val >= 0.95:
        first_95 = (method, idx)
    if val >= 0.99 and method not in seen_99_methods:
        seen_99_methods.add(method)
        if first_99 is None:
            first_99 = (method, idx)
        elif second_99 is None:
            second_99 = (method, idx)
        elif third_99 is None:
            third_99 = (method, idx)
    if first_95 and first_99 and second_99 and third_99:
        break

results_summary = {
    "first_95_method": first_95[0] if first_95 else None,
    "first_95_index": first_95[1] if first_95 else None,
    "first_99_method": first_99[0] if first_99 else None,
    "first_99_index": first_99[1] if first_99 else None,
    "second_99_method": second_99[0] if second_99 else None,
    "second_99_index": second_99[1] if second_99 else None,
    "third_99_index": third_99[1] if third_99 else None,
    "third_99_method": third_99[0] if third_99 else None
}



In [79]:

EPS = 1e-8
attack_types = ["Benign", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]

def normalize(p):
    p = np.asarray(p, dtype=float)
    return p / p.sum()

def kl_contributions(p, q):
    """
    Per-class contribution to KL(P || Q)
    """
    #p = normalize(p)
    #q = normalize(q)
    q = np.clip(q, EPS, None)
    return p * np.log(p / q)


In [80]:

matrices = {
    "First_95": mean_distr_95,
    "First_99": mean_distr_99_1,
    "Second_99": mean_distr_99_2,
    "Third_99": mean_distr_99_3
}

rows = []
#["Predictproba_single", "Predictproba_batch", "Random", "KU_single", "KU_batch", "QBC"]

for stage, distr_dict in matrices.items():
    # 'Random' is the baseline Q distribution
    p_random = distr_dict["Predictproba_single"]

    for method, p_method in distr_dict.items():
        if method == "Predictproba_single":
            continue

        # 1. Calculate the vector of contributions (what you currently have)
        contribs = abs(kl_contributions(p_method, p_random))
        contribs = np.delete(contribs, 6)
        # 2. Sum them up to get the single Total KL Divergence score
        total_kl = np.sum(contribs)

        rows.append({
            "Stage": stage,
            "Algorithm": method,
            "Total_KL_Divergence": total_kl
        })

# Create the simplified dataframe
df_total_kl = pd.DataFrame(rows)

# Save to CSV
csv_path_total = f"./KL_div_total_ppsingle/{start_type}_Total_KL_divergence_ppsingle.csv"
df_total_kl.to_csv(csv_path_total, index=False)

print(f"Saved Total KL Divergence table to: {csv_path_total}")
print(df_total_kl.head())

Saved Total KL Divergence table to: ./KL_div_total_ppsingle/Syn_Total_KL_divergence_ppsingle.csv
      Stage           Algorithm  Total_KL_Divergence
0  First_95            KU_batch            24.084535
1  First_95           KU_single             8.951309
2  First_95  Predictproba_batch            22.523906
3  First_95                 QBC            16.200380
4  First_95              Random            30.203029


/tmp/ipykernel_443931/3602243183.py:15: RuntimeWarning: divide by zero encountered in log
  return p * np.log(p / q)
/tmp/ipykernel_443931/3602243183.py:15: RuntimeWarning: invalid value encountered in multiply
  return p * np.log(p / q)
